In [1]:
import numpy as np

# Gradient Descent

In [5]:
class GradientDescent():
    def __init__(self):
        self.learning_rate = 0.1
        self.epoch = 10000
        self.tol =1e-5
        self.weights = None
        self.bias = 0
        self.loss_history = []
        self.patience = 20
    def predict(self , X):
        return (X @ self.weights) + self.bias
    @staticmethod
    def compute_cost(error ,m):
        return np.sum(error**2)/(2*m)
    def compute_gradients(self,X , error ,m):
        return (X.T @ error)/m , np.sum(error)/m
    def update_gradients(self ,dw , db):
        self.weights -= self.learning_rate*dw
        self.bias -= self.learning_rate*db
    def fit_batch_decent(self,X,y):
        m , n =X.shape
        self.weights = np.zeros((n,1))
        prev_cost =None
        count  = 0
        for _ in range(self.epoch):
            y_pred = self.predict(X)
            error = y_pred - y
            cost = GradientDescent.compute_cost(error , m)
            if prev_cost is not None and abs(prev_cost-cost) < self.tol:
                if count < self.patience:
                    count+=1
                else:
                    print("model converged")
                    break
            if prev_cost is not None and cost > prev_cost * 5:
                print("Training diverged.")
                break
            prev_cost = cost
            self.loss_history.append(cost)
            dw, db = self.compute_gradients(X,error,m)
            self.update_gradients(dw,db)
        return self
    def fit_stochastic_descent(self,X,y):
        m,n = X.shape
        self.weights = np.ones((n,1))
        prev_cost  = None
        for _ in range(self.epoch):
            idx = np.random.permutation(m)
            X_shuffle = X[idx]
            y_shuffle = y[idx]
            for i in range(m):
                x_cur = X_shuffle[i:i+1]
                y_cur = y_shuffle[i:i+1]
                y_cur_pred = self.predict(x_cur)
                error_cur = y_cur_pred - y_cur
                dw, db = self.compute_gradients(x_cur,error_cur,1)
                self.update_gradients(dw,db)
            y_pred = self.predict(X_shuffle)
            cost = GradientDescent.compute_cost(y_pred-y_shuffle,m)
            if prev_cost is not None and abs(prev_cost-cost) < self.tol:
                print("model converged")
                break
            if prev_cost is not None and cost > prev_cost * 5:
                print("Training diverged.")
                break
            prev_cost = cost
            self.loss_history.append(cost)
            return self
    def fit_mini_batch_descent(self,X,y,batch_size):
        m,n = X.shape
        self.weights = np.ones((n,1))
        prev_cost  = None
        for _ in range(self.epoch):
            idx = np.random.permutation(m)
            X_shuffle = X[idx]
            y_shuffle = y[idx]
            for i in range(0,m,batch_size):
                x_cur = X_shuffle[i:i+batch_size]
                y_cur = y_shuffle[i:i+batch_size]
                y_cur_pred = self.predict(x_cur)
                error_cur = y_cur_pred - y_cur
                dw, db = self.compute_gradients(x_cur,error_cur,x_cur.shape[0])
                self.update_gradients(dw,db)
            y_pred = self.predict(X_shuffle)
            cost = GradientDescent.compute_cost(y_pred-y_shuffle,m)
            if prev_cost is not None and abs(prev_cost-cost) < self.tol:
                print("model converged")
                break
            if prev_cost is not None and cost > prev_cost * 5:
                print("Training diverged.")
                break
            prev_cost = cost
            self.loss_history.append(cost)
        return self
    def fit_adagrad(self,X,y):
        m,n =X.shape
        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y size mismatch")        
        self.weights = np.zeros((n,1))
        self.bias = 0
        grad_sq_w = np.zeros((n,1))
        grad_sq_b = 0
        eps = 1e-8 ## for diving numerical stability doesnt add any value but avoids divided by zero
        prev_cost =None
        for _ in range(self.epoch):
            y_pred = self.predict(X)
            error = y_pred-y
            cost = GradientDescent.compute_cost(error)
            if prev_cost is not None and abs(prev_cost-cost) < self.tol:
                print("model converged")
                break
            if prev_cost is not None and cost > prev_cost * 5:
                print("Training diverged.")
                break
            prev_cost = cost
            self.loss_history.append(cost)
            dw , db = self.compute_gradients(X, error ,m)
            grad_sq_w += (dw**2)
            grad_sq_b += (db**2)
            self.weights -= (self.learning_rate*dw)/np.sqrt(grad_sq_w+eps)
            self.bias -= (self.learning_rate*db)/np.sqrt(grad_sq_b+eps)
        return self
    def fit_rmsprop(self,X,y):
        m,n =X.shape
        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y size mismatch")        
        self.weights = np.zeros((n,1))
        self.bias = 0
        rms_squared_w = np.zeros((n,1))
        rms_squared_b = 0
        eps = 1e-8 ## for adding it for numerical stability when divided by zero , doesnt add any intution
        prev_cost =None
        beta = 0.9
        for _ in range(self.epoch):
            y_pred = self.predict(X)
            error = y_pred-y
            cost = GradientDescent.compute_cost(error)
            if prev_cost is not None and abs(prev_cost-cost) < self.tol:
                print("model converged")
                break
            if prev_cost is not None and cost > prev_cost * 5:
                print("Training diverged.")
                break
            prev_cost = cost
            self.loss_history.append(cost)
            dw , db = self.compute_gradients(X, error ,m)
            rms_squared_w = (beta*rms_squared_w) + (1-beta)*(dw**2)
            rms_squared_b = (beta*rms_squared_b) + (1-beta)*(db**2)
            self.weights -= (self.learning_rate*dw)/np.sqrt(rms_squared_w+eps)
            self.bias -= (self.learning_rate*db)/np.sqrt(rms_squared_b+eps)
        return self

            

# momentum

In [10]:
class Momentum():
    def __init__(self,learning_rate , epoch):
        self.learning_rate = learning_rate
        self.epoch = epoch
        self.weights = None
        self.bias = 0
        self.velocity_w = None
        self.velocity_b = None
        self.beta = 0.9
        self.tol = 1e-5
        self.loss_history= []
    def predict(self,X):
        return (X @ self.weights) + self.bias
    @staticmethod
    def compute_cost(error ,m):
        return np.sum(error**2)/(2*m)
    def compute_velocity(self,X , error ,m):
        dw,db = (X.T @ error)/m , np.sum(error)/m
        self.velocity_w = (self.beta*self.velocity_w) - (self.learning_rate*dw) 
        self.velocity_b = (self.beta*self.velocity_b) - (self.learning_rate*db)  
    def update_parameters(self):
        self.weights += self.velocity_w
        self.bias += self.velocity_b
    def fit_momentum(self,X,y):
        m,n = X.shape
        if X.shape[0] != y.shape[0]:
            print("length mismatch")
            return -1
        self.weights = np.zeros((n,1))
        self.velocity_w = np.zeros((n,1))
        self.bias = 0
        self.velocity_b = 0
        prev_cost= None
        for _ in range(self.epoch):
            y_pred = self.predict(X)
            error = y_pred -y
            cost = Momentum.compute_cost(error,m)

            if prev_cost is not None: 
                if abs(prev_cost-cost) < self.tol:
                    print("model converged")
                    break
            if prev_cost is not None :
                if cost > prev_cost * 5:
                    print("Training diverged.")
                    break
            prev_cost = cost
            self.loss_history.append(cost)
            
            self.compute_velocity(X,error,m)
            self.update_parameters()            
        return self
    def fit_nesterov_momentum(self,X,y):
        m,n = X.shape
        if y.shape[0] != X.shape[0]:
            print("X, y length mismatch")
        self.weights = np.zeros((n,1))
        self.velocity_w = np.zeros((n,1))
        self.bias = 0
        self.velocity_b = 0
        
        prev_cost = None
        for _ in range(self.epoch):
           
            w_look = self.weights+self.beta * self.velocity_w
            b_look = self.bias + self.beta * self.velocity_b
            y_pred = X @ w_look + b_look
            error = y_pred-y
            cost = Momentum.compute_cost(error)
            if prev_cost is not None: 
                if abs(prev_cost-cost) < self.tol:
                    print("model converged")
                    break
            if prev_cost is not None :
                if cost > prev_cost * 5:
                    print("Training diverged.")
                    break
            prev_cost = cost
            self.loss_history.append(cost)
            self.compute_velocity()
            self.update_parameters()
        return self
    def fit_adam(self,X,y):
        if X.shape[0] != y.shape[0]:
            raise ValueError("length of X and y are not same")
        m,n = X.shape
        self.weights =np.zeros((n,1))
        self.bias = 0
        prev_cost =None
        eps = 1e-8

        self.velocity_w= np.zeros((n,1))
        self.velocity_b = 0
        beta_1 = 0.9

            
        beta_2 =0.999
        grad_sq_w = np.zeros((n,1))
        grad_sq_b = 0
        
        for i in range(1, self.epoch+1):
            y_pred = self.predict(X)
            error = y_pred-y
            cost= Momentum.compute_cost(error,m)
            if prev_cost is not None: 
                    if abs(prev_cost-cost) < self.tol:
                        print("model converged")
                        break
            if prev_cost is not None :
                    if cost > prev_cost * 5:
                        print("Training diverged.")
                        break
            prev_cost = cost
            self.loss_history.append(cost)
            

            dw, db = (X.T @ error)/m , np.sum(error)/m
    
            self.velocity_w = beta_1* self.velocity_w+(1-beta_1)*dw
            m_hat_w = self.velocity_w/(1-beta_1**i)
            self.velocity_b = (beta_1 * self.velocity_b +(1-beta_1)*db)
            m_hat_b = self.velocity_b/(1-beta_1**i)
    
            grad_sq_w = beta_2*grad_sq_w + (1-beta_2)*dw**2
            v_hat_w = grad_sq_w/(1-beta_2**i)
            grad_sq_b = beta_2*grad_sq_b + (1-beta_2)*db**2
            v_hat_b = grad_sq_b/(1-beta_2**i)
    
            #update parameters
            self.weights  = self.weights -((self.learning_rate* m_hat_w)/(np.sqrt(v_hat_w)+eps))
            self.bias = self.bias - ((self.learning_rate *m_hat_b)/(np.sqrt(v_hat_b) + eps))
        return self
    



            
            
    
    

## Learning Rate Finder

In [13]:
class LearningRateFinder():
    def __init__(self ,start_lr =1e-3,end_lr = 10,n_iter = 1000):
        self.start_lr = start_lr
        self.end_lr = end_lr
        self.lr = start_lr
        self.n_iter = n_iter
        self.weights = None
        self.bias = 0
        self.lr_list = None
        self.loss_list = None
    def predict(self,X):
        return (X @ self.weights) + self.bias
    def compute_gradients(self,X,error,m):
        return (X.T @ error)/m , np.sum(error)/m
    def update_parameters(self,dw,db):
        self.weights = self.weights - self.lr * dw
        self.bias = self.bias -self.lr*db
    def find(self ,X,y):
        multiplier = (self.end_lr/self.start_lr)**(1/self.n_iter)
        self.lr = self.start_lr
        m,n = X.shape
        self.weights =np.zeros((n,1))
        self.bias = 0
        best_loss =np.inf
        self.lr_list = []
        self.loss_list  = []
        for _ in range(self.n_iter):
            y_pred = self.predict(X)
            error = y_pred - y
            loss = np.sqrt(np.mean(error**2))
            if np.isnan(loss) or np.isinf(loss):
                print("Training diverged.")
                break

            if loss > 4 * best_loss:
                print("Loss exploded.")
                break
            best_loss = min(loss,best_loss)
            
            dw,db = self.compute_gradients(X,error,m)
            self.update_parameters(dw,db)
            self.loss_list.append(loss)
            self.lr_list.append(self.lr)
            self.lr *= multiplier
        return self
            
            
        
        
        
        
        
        

# Base Class Optimizer

In [14]:
class OptimizerBase():
    def __init__(self,learning_rate =0.1 , epoch=1000,tol = 1e-5,patience = 20):
        self.learning_rate =learning_rate
        self.epoch = epoch
        self.tol =tol
        self.weights = None
        self.bias = 0
        self.loss_history = None
        self.patience = patience
        
    def predict(self , X):
        return (X @ self.weights) + self.bias
    def compute_cost(self,error):
        return np.mean(error**2)/2
    def compute_gradients(self,X , error):
        m = X.shape[0]
        return (X.T @ error)/m , np.sum(error)/m
    def early_stopping(self,cost, prev_cost,count):
        stop=False
        if prev_cost is not None and abs(prev_cost-cost) < self.tol:
            if count < self.patience:
                count+=1
            else:
                count=0
                stop =True
        else:
            count=0
        return stop , count
    def init_parameters(self,n):
        self.weights = np.zeros((n,1))
        self.bias= 0
    def train_epoch(self,X,y):
        raise NotImplementedError
    def update_parameters(self,dw,db):
        raise NotImplementedError
        
    def fit(self,X,y):
        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y length mismatch")
        m,n =X.shape
        count = 0
        self.loss_history=[]
        prev_cost = None
        self.init_parameters(n)
        for  _ in range(self.epoch):
            self.train_epoch(X,y)
            y_pred = self.predict(X)
            error = y_pred - y
            cost = self.compute_cost(error)
            stop, count =self.early_stopping(cost, prev_cost,count)
            if stop:
                print("model converged")
                break
            prev_cost=cost
            self.loss_history.append(cost)
        return self

            
            

## Batch Gradient Descent

In [15]:
class BatchGD(OptimizerBase):
    def __init__(self,learning_rate=0.01,epoch =1000,tol=1e-5,patience=20):
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience
        )
    def train_epoch(self,X,y):
        error = self.predict(X) - y
        dw,db = self.compute_gradients(X,error)
        self.update_parameters(dw,db)
    def update_parameters(self,dw,db):
        self.weights -= self.learning_rate*dw
        self.bias -= self.learning_rate*db



## Stochastic Gradient Descent

In [16]:
class SGD(OptimizerBase):
    def __init__(self,learning_rate=0.01,epoch =1000,tol=1e-5,patience=20):
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience
        )
    def train_epoch(self,X,y):
        m= X.shape[0]
        idx = np.random.permutation(m)
        X_shuffle = X[idx]
        y_shuffle =y[idx]
        for i in range(m):
            Xi = X_shuffle[i:i+1]
            yi = y_shuffle[i:i+1]
            yi_pred = self.predict(Xi)
            error = yi_pred - yi
            dwi,dbi = self.compute_gradients(Xi,error)
            self.update_parameters(dwi,dbi)
    def update_parameters(self,dw,db):
        self.weights -= self.learning_rate*dw
        self.bias -= self.learning_rate*db

        
        


## Mini Batch Gradient Descent

In [17]:
class MiniBatchGD(OptimizerBase):
    def __init__(self,learning_rate=0.01,epoch =1000,tol=1e-5,patience=20,batch_size=32):
        self.batch_size = batch_size
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience
        )
    def train_epoch(self,X,y):
        m= X.shape[0]
        idx = np.random.permutation(m)
        X_shuffle = X[idx]
        y_shuffle =y[idx]
        for i in range(0,m,self.batch_size):
            X_batch = X_shuffle[i:i+self.batch_size]
            y_batch = y_shuffle[i:i+self.batch_size]
            y_pred = self.predict(X_batch)
            error = y_pred - y_batch
            dw,db = self.compute_gradients(X_batch,error)
            self.update_parameters(dw,db)
    def update_parameters(self,dw,db):
        self.weights -= self.learning_rate*dw
        self.bias -= self.learning_rate*db

## Momentum

In [15]:
class Momentum(BatchGD):
    def __init__(self,learning_rate=0.01,epoch =1000,tol=1e-5,patience=20,beta=0.9):
        self.beta = beta
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience
        )
    def init_parameters(self,n):
        super().init_parameters(n)
        self.velocity_w = np.zeros((n,1))
        self.velocity_b = 0.0
    def update_parameters(self,dw,db):
        self.velocity_w = (self.beta * self.velocity_w)  - (self.learning_rate*dw)
        self.velocity_b = (self.beta * self.velocity_b) - (self.learning_rate*db)
        self.weights += self.velocity_w
        self.bias += self.velocity_b

        

## Nesterov Adaptive Descent

In [18]:
class NAD(Momentum):
    def __init__(self,learning_rate=0.01,epoch =1000,tol=1e-5,patience=20,beta=0.9):
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience,
            beta =beta
        )
    def train_epoch(self,X,y):
        w_look =self.weights + (self.beta* self.velocity_w)
        b_look = self.bias + (self.beta*self.velocity_b)
        y_look = (X @ w_look) + b_look
        error = y_look - y
        dw,db = self.compute_gradients(X,error)
        self.update_parameters(dw,db)

## Adaptive Gradient Descent

In [19]:
class AdaGrad(BatchGD):
    def __init__(self,learning_rate=0.01,epoch =1000,tol=1e-5,patience=20,eps = 1e-9):
        self.eps = eps
        self.grad_sq_w = None
        self.grad_sq_b = None
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience
        )
    def init_parameters(self,n):
        super().init_parameters(n)
        self.grad_sq_w = np.zeros((n,1))
        self.grad_sq_b = 0.0
    def update_grad_sq(self, dw,db):
        self.grad_sq_w += dw**2
        self.grad_sq_b += db**2
    def update_parameters(self,dw,db):
        self.weights -= (self.learning_rate*dw)/np.sqrt(self.grad_sq_w+self.eps)
        self.bias -= (self.learning_rate*db)/np.sqrt(self.grad_sq_b+self.eps)
        


## RMSProp

In [20]:
class RMSProp(AdaGrad):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience= 20,eps =1e-9,beta=0.9):
        self.beta = beta
        super().__init__(learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience,
            eps=eps)
    def update_grad_sq(self,dw,db):
        self.grad_sq_w = (self.beta*self.grad_sq_w) + ((1-self.beta)*(dw**2))
        self.grad_sq_b = (self.beta*self.grad_sq_b) + ((1-self.beta)*(db**2))


## Adam

In [21]:
class Adam(BatchGD):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience=20,eps=1e-9,beta_1=0.9,beta_2=0.999):
        self.eps = eps
        self.beta_1=beta_1
        self.beta_2=beta_2
        super().__init__(learning_rate= learning_rate,epoch=epoch,tol=tol,patience=patience)
    def init_parameters(self,n):
        self.t =0
        self.velocity_w = np.zeros((n,1))
        self.velocity_b = 0.0

        self.grad_sq_w = np.zeros((n,1))
        self.grad_sq_b = 0.0
        
        super().init_parameters(n)
    def compute_momentum(self,dw,db):
        self.velocity_w = (self.beta_1*self.velocity_w) + ((1-self.beta_1)*dw)
        self.velocity_b = (self.beta_1*self.velocity_b) + ((1-self.beta_1)*db)
    def compute_grad_sq(self,dw,db):
        self.grad_sq_w = (self.beta_2*self.grad_sq_w) +((1-self.beta_2)*(dw**2))
        self.grad_sq_b = (self.beta_2*self.grad_sq_b) + ((1-self.beta_2)*(db**2))
    def update_parameters(self,dw,db):
        self.compute_momentum(dw,db)
        self.compute_grad_sq(dw,db)
        self.t+=1
        bcm_w = self.velocity_w/(1-(self.beta_1**self.t))
        bcm_b = self.velocity_b/(1-(self.beta_1**self.t))

        bcsq_w = self.grad_sq_w/(1-(self.beta_2**self.t))
        bcsq_b = self.grad_sq_b/(1-(self.beta_2**self.t))

        self.weights -= self.learning_rate*bcm_w/np.sqrt(bcsq_w+self.eps)
        self.bias -= self.learning_rate*bcm_b/np.sqrt(bcsq_b+self.eps)
        

        
        
        
        

## Momentum with extend of Minibatch Gradient descent

In [22]:
class MomentumMinBatchGD(MiniBatchGD):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience=20,batch_size = 32,beta = 0.9):
        self.beta = beta
        super().__init__(learning_rate =learning_rate,epoch=epoch,tol =tol,batch_size=batch_size,patience=patience)
    def init_parameters(self,n):
        super().init_parameters(n)
        self.velocity_w = np.zeros((n,1))
        self.velocity_b = 0.0
    def update_velocity(self,dw,db):
        self.velocity_w = (self.beta *self.velocity_w) -(self.learning_rate*dw)
        self.velocity_b = (self.beta * self.velocity_b)- (self.learning_rate*db)
    def update_parameters(self,dw,db):
        self.update_velocity(dw,db)
        self.weights += self.velocity_w
        self.bias += self.velocity_b

## Nesterov with extension of Mini batch gradient descent

In [25]:
class NAD(MomentumMinBatchGD):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience=20,batch_size = 32,beta = 0.9):
        super().__init__(learning_rate =learning_rate,epoch=epoch,tol =tol,batch_size=batch_size,patience=patience,beta=beta)
    def train_epoch(self,X,y):
        m= X.shape[0]
        idx = np.random.permutation(m)
        X_shuffle = X[idx]
        y_shuffle =y[idx]
        for i in range(0,m,self.batch_size):
            X_batch = X_shuffle[i:i+self.batch_size]
            y_batch = y_shuffle[i:i+self.batch_size]
            w_look =self.weights + (self.beta* self.velocity_w)
            b_look =self.bias + (self.beta*self.velocity_b)
            y_pred = (X_batch @ w_look) + b_look
            error = y_pred - y_batch
            dw,db = self.compute_gradients(X_batch,error)
            self.update_parameters(dw,db)


## Adagrad with extension of mini batch gradient descent

In [22]:
class AdagradMiniBatch(MiniBatchGD):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience=20,batch_size = 32,eps=1e-9):
        self.eps = eps
        super().__init__(learning_rate =learning_rate,epoch=epoch,tol =tol,batch_size=batch_size,patience=patience)
    def init_parameters(self,n):
        super().init_parameters(n)
        self.grad_sq_w = np.zeros((n,1))
        self.grad_sq_b = 0.0
    def update_grad_sq(self,dw,db):
        self.grad_sq_w = self.grad_sq_w+(dw**2)
        self.grad_sq_b = self.grad_sq_b+(db**2)
    def update_parameters(self,dw,db):
        self.update_grad_sq(dw,db)
        self.weights -= (self.learning_rate*dw)/np.sqrt(self.grad_sq_w+self.eps) 
        self.bias -= (self.learning_rate*db)/np.sqrt(self.grad_sq_b+self.eps)

## RMSProp with extension of adagrad minibatch gradient descent

In [23]:
class RMSProp(AdagradMiniBatch):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience=20,batch_size = 32,eps=1e-9,beta=0.9):
        self.beta = beta
        super().__init__(learning_rate =learning_rate,epoch=epoch,tol =tol,batch_size=batch_size,patience=patience,eps=eps)
    def update_grad_sq(self,dw,db):
        self.grad_sq_w = self.beta * self.grad_sq_w + ((1-self.beta)*(self.grad_sq_w**2))
        self.grad_sq_b = self.beta * self.grad_sq_b + ((1-self.beta)* (self.grad_sq_b**2))
    

    

## Adam with extension of mini batch gradient descent

In [24]:
class Adam(MiniBatchGD):
    def __init__(self,learning_rate=0.01,epoch=1000,tol=1e-5,patience=20,batch_size=32,eps=1e-9,beta_1=0.99,beta_2=0.9):
        self.eps =eps
        self.beta_1=beta_1
        self.beta_2 =beta_2
        super().__init__(
            learning_rate=learning_rate,
            epoch=epoch,
            tol=tol,
            patience=patience,
            batch_size=batch_size
            )
    def init_parameters(self,n):
        super().init_parameters(n)
        self.velocity_w = np.zeros((n,1))
        self.velocity_b = 0.0

        self.grad_sq_w = np.zeros((n,1))
        self.grad_sq_b = 0.0

        self.t = 0
    def update_velocity(self,dw,db):
        self.velocity_w = self.beta_1 * self.velocity_w + (1-self.beta_1)*dw
        self.velocity_b = self.beta_1* self.velocity_b + (1-self.beta_1)*db

    def update_grad_sq(self , dw,db):
        self.grad_sq_w = self.beta_2* self.grad_sq_w + (1-self.beta_2)*(dw**2) 
        self.grad_sq_b = self.beta_2*self.grad_sq_b +(1-self.beta_2)*(db**2)

    def update_parameters(self,dw,db):
        self.t +=1
        self.update_velocity(dw,db)
        self.update_grad_sq(dw,db)
        m_hat_w =self.velocity_w / (1-(self.beta_1)**self.t)
        m_hat_b = self.velocity_b / (1-(self.beta_1)**self.t)

        v_hat_w = self.grad_sq_w / (1-(self.beta_2**self.t))
        v_hat_b = self.grad_sq_b / (1-(self.beta_2**self.t))

        self.weights -= m_hat_w*self.learning_rate/np.sqrt(v_hat_w+self.eps)
        self.bias -= m_hat_b*self.learning_rate/np.sqrt(v_hat_b+self.eps)
        
        